In [ ]:
# Colab Spark Setup: Single Cell

# 1. Install PySpark (includes Spark + Hadoop + Py4J)
!pip install --quiet pyspark==3.5.1

# 2. Import Spark and create session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ColabSparkSetup") \
    .getOrCreate()



+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
+-------+---+



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sobhanmoosavi/us-weather-events")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/us-weather-events


In [ ]:
import os

# List dataset files
os.listdir(path)

['WeatherEvents_Jan2016-Dec2022.csv']

In [ ]:
# Read CSV into Spark
df = spark.read.csv(f"{path}/WeatherEvents_Jan2016-Dec2022.csv", header=True, inferSchema=True)

# Show first 5 rows
df.show(5)

+-------+----+--------+-------------------+-------------------+-----------------+-----------+-----------+-----------+-----------+--------+--------+-----+-------+
|EventId|Type|Severity|     StartTime(UTC)|       EndTime(UTC)|Precipitation(in)|   TimeZone|AirportCode|LocationLat|LocationLng|    City|  County|State|ZipCode|
+-------+----+--------+-------------------+-------------------+-----------------+-----------+-----------+-----------+-----------+--------+--------+-----+-------+
|    W-1|Snow|   Light|2016-01-06 23:14:00|2016-01-07 00:34:00|              0.0|US/Mountain|       K04V|    38.0972|  -106.1689|Saguache|Saguache|   CO|  81149|
|    W-2|Snow|   Light|2016-01-07 04:14:00|2016-01-07 04:54:00|              0.0|US/Mountain|       K04V|    38.0972|  -106.1689|Saguache|Saguache|   CO|  81149|
|    W-3|Snow|   Light|2016-01-07 05:54:00|2016-01-07 15:34:00|             0.03|US/Mountain|       K04V|    38.0972|  -106.1689|Saguache|Saguache|   CO|  81149|
|    W-4|Snow|   Light|2016-

In [ ]:
# Check total rows
total_rows = df.count()
print("Total rows in dataset:", total_rows)

Total rows in dataset: 8627181


In [ ]:
from pyspark.sql.functions import col, year, to_timestamp
# Convert StartTime(UTC) to timestamp type
df = df.withColumn("StartTime", to_timestamp(col("StartTime(UTC)")))

# Filter data for the year 2017
df_2017 = df.filter(year(col("StartTime")) == 2017)

# Show total rows and sample data
print("Total rows for 2017:", df_2017.count())
df_2017.show(5)

Total rows for 2017: 1227786
+-------+----+--------+-------------------+-------------------+-----------------+-----------+-----------+-----------+-----------+--------+--------+-----+-------+-------------------+
|EventId|Type|Severity|     StartTime(UTC)|       EndTime(UTC)|Precipitation(in)|   TimeZone|AirportCode|LocationLat|LocationLng|    City|  County|State|ZipCode|          StartTime|
+-------+----+--------+-------------------+-------------------+-----------------+-----------+-----------+-----------+-----------+--------+--------+-----+-------+-------------------+
|  W-548|Snow|   Light|2017-01-01 12:30:00|2017-01-01 12:50:00|              0.0|US/Mountain|       K04V|    38.0972|  -106.1689|Saguache|Saguache|   CO|  81149|2017-01-01 12:30:00|
|  W-549| Fog|  Severe|2017-01-01 12:50:00|2017-01-01 13:10:00|              0.0|US/Mountain|       K04V|    38.0972|  -106.1689|Saguache|Saguache|   CO|  81149|2017-01-01 12:50:00|
|  W-550|Snow|   Light|2017-01-01 15:50:00|2017-01-01 16:30:0

In [ ]:
from pyspark.sql.functions import col, sum
# Count nulls in each column
missing_data = df_2017.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_2017.columns
])

missing_data.show()

+-------+----+--------+--------------+------------+-----------------+--------+-----------+-----------+-----------+----+------+-----+-------+---------+
|EventId|Type|Severity|StartTime(UTC)|EndTime(UTC)|Precipitation(in)|TimeZone|AirportCode|LocationLat|LocationLng|City|County|State|ZipCode|StartTime|
+-------+----+--------+--------------+------------+-----------------+--------+-----------+-----------+-----------+----+------+-----+-------+---------+
|      0|   0|       0|             0|           0|                0|       0|          0|          0|          0|2464|     0|    0|   9983|        0|
+-------+----+--------+--------------+------------+-----------------+--------+-----------+-----------+-----------+----+------+-----+-------+---------+



In [ ]:
df_cleaned = df_2017.dropna(subset=["City"])

In [ ]:
import kagglehub

# Download via kagglehub
path = kagglehub.dataset_download("gabrielluizone/us-domestic-flights-delay-prediction-2013-2018")
print("Dataset path:", path)

Dataset path: /kaggle/input/us-domestic-flights-delay-prediction-2013-2018


In [ ]:
import os
print(os.listdir(path))

['flight_delay_predict.csv', 'csv_flight']


In [ ]:
from pyspark.sql.functions import to_date, col, year

# Read CSV into Spark
flights_df = spark.read.csv(f"{path}/flight_delay_predict.csv", header=True, inferSchema=True)
flights_df.show(5)

+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+
|is_delay|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Origin|OriginState|Dest|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDelay|ArrDelayMinutes|AirTime|
+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+
|     1.0|2014|      1|    1|         1|        3|2014-01-01|               UA|   LAX|         CA| ORD|       IL|       900|      0.0|     0.0|  1744.0|            7|    43.0|           43.0|  218.0|
|     0.0|2014|      1|    1|         1|        3|2014-01-01|               AA|   IAH|         TX| DFW|       TX|      1750|      0.0|     0.0|   224.0|            1|     2.0|            2.0|   50.0|


In [ ]:
from pyspark.sql.functions import col

# Filter rows where Year = 2017
flights_2017 = flights_df.filter(col("Year") == 2017)

# Check total rows
print("Total flights in 2017:", flights_2017.count())

# Preview data
flights_2017.show(5)

Total flights in 2017: 344450
+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+
|is_delay|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Origin|OriginState|Dest|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDelay|ArrDelayMinutes|AirTime|
+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+
|     0.0|2017|      1|    1|         1|        7|2017-01-01|               AA|   DEN|         CO| CLT|       NC|      1011|      0.0|     0.0|  1337.0|            6|    -8.0|            0.0|  159.0|
|     1.0|2017|      1|    1|         1|        7|2017-01-01|               AA|   ATL|         GA| ORD|       IL|       959|      0.0|     0.0|   606.0|            3|   1

In [ ]:
from pyspark.sql.functions import col, sum

missing_summary = flights_2017.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in ["Origin", "Dest", "FlightDate"]
])

missing_summary.show()

+------+----+----------+
|Origin|Dest|FlightDate|
+------+----+----------+
|     0|   0|         0|
+------+----+----------+



In [ ]:
from pyspark.sql.functions import expr, to_date

# Convert ICAO -> IATA
weather_ready = df_2017.withColumn("IATA", expr("substring(AirportCode, 2, 3)"))

# Extract date for join
weather_ready = weather_ready.withColumn("DATE", to_date("StartTime"))

In [ ]:
combined_df = flights_2017.join(
    weather_ready,
    (flights_2017.Origin == weather_ready.IATA) &
    (flights_2017.FlightDate == weather_ready.DATE),
    "left"
)

print("Combined rows:", combined_df.count())
combined_df.show(5)

Combined rows: 624694
+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+---------+----+--------+-------------------+-------------------+-----------------+----------+-----------+-----------+-----------+------+------+-----+-------+-------------------+----+----------+
|is_delay|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Origin|OriginState|Dest|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDelay|ArrDelayMinutes|AirTime|  EventId|Type|Severity|     StartTime(UTC)|       EndTime(UTC)|Precipitation(in)|  TimeZone|AirportCode|LocationLat|LocationLng|  City|County|State|ZipCode|          StartTime|IATA|      DATE|
+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-

In [ ]:
from pyspark.sql.functions import col, sum

null_counts = combined_df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in combined_df.columns
])
null_counts.show(truncate=False)

+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+-------+------+--------+--------------+------------+-----------------+--------+-----------+-----------+-----------+------+------+------+-------+---------+------+------+
|is_delay|Year|Quarter|Month|DayofMonth|DayOfWeek|FlightDate|Reporting_Airline|Origin|OriginState|Dest|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDelay|ArrDelayMinutes|AirTime|EventId|Type  |Severity|StartTime(UTC)|EndTime(UTC)|Precipitation(in)|TimeZone|AirportCode|LocationLat|LocationLng|City  |County|State |ZipCode|StartTime|IATA  |DATE  |
+--------+----+-------+-----+----------+---------+----------+-----------------+------+-----------+----+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+-------+------+--------+--------------+------------+-------------

In [ ]:
combined_filled = combined_df.fillna({
    "EventId": "NoEvent",
    "Type": "NoEvent",
    "Severity": "None",
    "Precipitation(in)": 0.0
})

In [ ]:
columns = combined_all.columns
seen = {}
new_columns = []

for col in columns:
    lower_col = col.lower()  # normalize to lowercase for duplicate check
    if lower_col in seen:
        seen[lower_col] += 1
        new_columns.append(f"{col}_{seen[lower_col]}")  # add suffix to make unique
    else:
        seen[lower_col] = 0
        new_columns.append(col)

combined_all_unique = combined_all.toDF(*new_columns)

print(combined_all_unique.columns)  # Verify unique columns


['is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'date', 'Reporting_Airline', 'origin', 'OriginState', 'destination', 'DestState', 'CRSDepTime', 'Cancelled', 'Diverted', 'Distance', 'DistanceGroup', 'ArrDelay', 'ArrDelayMinutes', 'AirTime', 'EventId', 'weather_type', 'severity', 'StartTime(UTC)', 'EndTime(UTC)', 'precipitation', 'TimeZone', 'AirportCode', 'LocationLat', 'LocationLng', 'City', 'County', 'State', 'ZipCode', 'StartTime', 'IATA', 'DATE_1']


In [ ]:
from pyspark.sql.functions import dayofyear, expr, hour, minute, second

# 1️⃣ First select only unique columns to remove ambiguity
combined_unique = combined_all.toDF(*[
    'is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'date',
    'Reporting_Airline', 'origin', 'OriginState', 'destination', 'DestState',
    'CRSDepTime', 'Cancelled', 'Diverted', 'Distance', 'DistanceGroup',
    'ArrDelay', 'ArrDelayMinutes', 'AirTime', 'EventId', 'weather_type',
    'severity', 'StartTime(UTC)', 'EndTime(UTC)', 'precipitation', 'TimeZone',
    'AirportCode', 'LocationLat', 'LocationLng', 'City', 'County', 'State',
    'ZipCode', 'StartTime', 'IATA', 'DATE_1'
])

# 2️⃣ Compute day of year
df_shifted = combined_unique.withColumn("day_of_year", dayofyear("date"))

# 3️⃣ Create 2024 date based on calendar
df_shifted = df_shifted.withColumn(
    "date_2024",
    expr("date_add(to_date('2024-01-01'), day_of_year - 1)")
)

# 4️⃣ Shift StartTime and EndTime to 2024, preserving time
df_shifted = df_shifted.withColumn(
    "StartTime_2024",
    expr("timestampadd(DAY, day_of_year - 1, make_timestamp(2024,1,1,hour(`StartTime(UTC)`),minute(`StartTime(UTC)`),second(`StartTime(UTC)`)))")
)
df_shifted = df_shifted.withColumn(
    "EndTime_2024",
    expr("timestampadd(DAY, day_of_year - 1, make_timestamp(2024,1,1,hour(`EndTime(UTC)`),minute(`EndTime(UTC)`),second(`EndTime(UTC)`)))")
)

# 5️⃣ Replace old columns with shifted columns
combined_2024 = (
    df_shifted
    .drop("date", "StartTime(UTC)", "EndTime(UTC)", "day_of_year")
    .withColumnRenamed("date_2024", "date")
    .withColumnRenamed("StartTime_2024", "StartTime(UTC)")
    .withColumnRenamed("EndTime_2024", "EndTime(UTC)")
)

combined_2024.show(3)


+--------+----+-------+-----+----------+---------+-----------------+------+-----------+-----------+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+---------+------------+--------+-------------+----------+-----------+-----------+-----------+-------+-------+-----+-------+-------------------+----+----------+----------+-------------------+-------------------+
|is_delay|Year|Quarter|Month|DayofMonth|DayOfWeek|Reporting_Airline|origin|OriginState|destination|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDelay|ArrDelayMinutes|AirTime|  EventId|weather_type|severity|precipitation|  TimeZone|AirportCode|LocationLat|LocationLng|   City| County|State|ZipCode|          StartTime|IATA|    DATE_1|      date|     StartTime(UTC)|       EndTime(UTC)|
+--------+----+-------+-----+----------+---------+-----------------+------+-----------+-----------+---------+----------+---------+--------+--------+-------------+--------+---------------+-

In [ ]:
# 1️⃣ Drop columns related to 2017
columns_to_drop = ["DATE_1", "StartTime", "Year"]
combined_2024_cleaned = combined_2024.drop(*columns_to_drop)

# 2️⃣ Verify remaining columns
print("Columns after cleanup:", combined_2024_cleaned.columns)

# 3️⃣ Preview cleaned dataset
combined_2024_cleaned.show(3)


Columns after cleanup: ['is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'origin', 'OriginState', 'destination', 'DestState', 'CRSDepTime', 'Cancelled', 'Diverted', 'Distance', 'DistanceGroup', 'ArrDelay', 'ArrDelayMinutes', 'AirTime', 'EventId', 'weather_type', 'severity', 'precipitation', 'TimeZone', 'AirportCode', 'LocationLat', 'LocationLng', 'City', 'County', 'State', 'ZipCode', 'IATA', 'date', 'StartTime(UTC)', 'EndTime(UTC)']
+--------+-------+-----+----------+---------+-----------------+------+-----------+-----------+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+---------+------------+--------+-------------+----------+-----------+-----------+-----------+-------+-------+-----+-------+----+----------+-------------------+-------------------+
|is_delay|Quarter|Month|DayofMonth|DayOfWeek|Reporting_Airline|origin|OriginState|destination|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDe

In [ ]:
from pyspark.sql.functions import to_date, date_format

# Split and drop original timestamp columns
df_split = combined_2024_cleaned.withColumn("StartDate", to_date("StartTime(UTC)")) \
                       .withColumn("StartTimeOnly", date_format("StartTime(UTC)", "HH:mm:ss")) \
                       .withColumn("EndDate", to_date("EndTime(UTC)")) \
                       .withColumn("EndTimeOnly", date_format("EndTime(UTC)", "HH:mm:ss")) \
                       .drop("StartTime(UTC)", "EndTime(UTC)")


In [ ]:
print(df_split.columns)  # Verify unique columns

['is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'origin', 'OriginState', 'destination', 'DestState', 'CRSDepTime', 'Cancelled', 'Diverted', 'Distance', 'DistanceGroup', 'ArrDelay', 'ArrDelayMinutes', 'AirTime', 'EventId', 'weather_type', 'severity', 'precipitation', 'TimeZone', 'AirportCode', 'LocationLat', 'LocationLng', 'City', 'County', 'State', 'ZipCode', 'IATA', 'date', 'StartDate', 'StartTimeOnly', 'EndDate', 'EndTimeOnly']


In [ ]:
from pyspark.sql.functions import when

# Add Season column based on Month
df_with_season = df_split.withColumn(
    "Season",
    when(df_split.Month.isin(12, 1, 2), "Winter")
    .when(df_split.Month.isin(3, 4, 5), "Spring")
    .when(df_split.Month.isin(6, 7, 8), "Summer")
    .otherwise("Fall")  # 9,10,11
)


In [ ]:
df_with_season.show(3)

+--------+-------+-----+----------+---------+-----------------+------+-----------+-----------+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+---------+------------+--------+-------------+----------+-----------+-----------+-----------+-------+-------+-----+-------+----+----------+----------+-------------+----------+-----------+------+
|is_delay|Quarter|Month|DayofMonth|DayOfWeek|Reporting_Airline|origin|OriginState|destination|DestState|CRSDepTime|Cancelled|Diverted|Distance|DistanceGroup|ArrDelay|ArrDelayMinutes|AirTime|  EventId|weather_type|severity|precipitation|  TimeZone|AirportCode|LocationLat|LocationLng|   City| County|State|ZipCode|IATA|      date| StartDate|StartTimeOnly|   EndDate|EndTimeOnly|Season|
+--------+-------+-----+----------+---------+-----------------+------+-----------+-----------+---------+----------+---------+--------+--------+-------------+--------+---------------+-------+---------+------------+--------+--------

In [ ]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime


In [ ]:
def preprocess_test_data(df):
    df = df.copy()

    # Ensure FL_DATE is datetime
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])
    df['MONTH'] = df['FL_DATE'].dt.month
    df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek
    df['HOUR_OF_DAY'] = df['CRS_DEP_TIME'] // 100  # Extract hour component

    # Season mapping
    season_map = {
        12: 'Winter', 1: 'Winter', 2: 'Winter',
        3: 'Spring', 4: 'Spring', 5: 'Spring',
        6: 'Summer', 7: 'Summer', 8: 'Summer',
        9: 'Fall', 10: 'Fall', 11: 'Fall'
    }
    df['SEASON'] = df['MONTH'].map(season_map)

    # Time-of-day categories
    bins = [0, 600, 1200, 1800, 2400]
    labels = ['Early_Morning', 'Morning', 'Afternoon', 'Evening']
    df['TIME_CATEGORY'] = pd.cut(df['CRS_DEP_TIME'], bins=bins, labels=labels, right=False)

    # Severity score mapping
    severity_map = {
        'Light': 1,
        'Moderate': 2,
        'Heavy': 3,
        'Severe': 4,
        'Unknown': 1
    }
    df['SEVERITY_SCORE'] = df['Severity'].map(severity_map)

    # Precipitation categories
    df['PRECIP_CAT'] = pd.cut(
        df['Precipitation(in)'],
        bins=[-1, 0.01, 0.1, 0.5, float('inf')],
        labels=['None', 'Light', 'Moderate', 'Heavy']
    )

    # Explicit weather severity flags
    df['HEAVY_RAIN'] = ((df['Type'] == 'Rain') &
                        (df['Severity'].isin(['Heavy', 'Severe']))).astype(int)
    df['SNOW_STORM'] = ((df['Type'] == 'Snow') &
                        (df['Severity'].isin(['Moderate', 'Heavy', 'Severe']))).astype(int)

    # Select the same features used for training
    features = [
        'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'MONTH', 'DAY_OF_WEEK', 'HOUR_OF_DAY',
        'SEASON', 'TIME_CATEGORY', 'Type', 'Severity', 'SEVERITY_SCORE',
        'Precipitation(in)', 'PRECIP_CAT', 'HEAVY_RAIN', 'SNOW_STORM'
    ]

    return df[features]


In [ ]:
# Example: Load test data
test_df = pd.read_csv("combined_2024_final.csv")

# Preprocess
X_test_processed = preprocess_test_data(test_df)


KeyError: 'FL_DATE'